In [108]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import f_regression, RFE
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
file_path = r"C:/Users/Dalbir/Downloads/Trends-Forecasting-Analytics-MLOps-Vertex-AI/data/processed_file/sales_featured.parquet"
df = pd.read_parquet(file_path)

In [109]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185686 entries, 0 to 185685
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   order_id          185686 non-null  string        
 1   product           185686 non-null  string        
 2   quantity_ordered  185686 non-null  int64         
 3   price_each        185686 non-null  float64       
 4   order_date        185686 non-null  datetime64[ns]
 5   purchase_address  185686 non-null  string        
 6   month             185686 non-null  int64         
 7   sales             185686 non-null  float64       
 8   city              185686 non-null  category      
 9   hour              185686 non-null  int64         
 10  year              185686 non-null  UInt32        
 11  week              185686 non-null  UInt32        
 12  day               185686 non-null  UInt32        
 13  day_name          185686 non-null  string        
 14  week

In [110]:
df["city"]=df["city"].astype("string")

In [111]:
categorical_cols = ['city', 'product', 'weekday_weekend']
drop_cols = ['order_id', 'order_date', 'sales', 'purchase_address', 'day_name', 'order_size']
numeric_cols = [col for col in df.columns if col not in categorical_cols + drop_cols]

X_num = df[numeric_cols]
y = df['sales']
kf = KFold(n_splits=5, shuffle=True, random_state=42)
global_mean = y.mean()

# Mean encode product
df['product_mean_encoded'] = np.nan
for tr, val in kf.split(df):
    means = df.iloc[tr].groupby('product')['sales'].mean()
    df.loc[val, 'product_mean_encoded'] = df.iloc[val]['product'].map(means)
df['product_mean_encoded'].fillna(global_mean, inplace=True)

# Mean encode city
df['city_mean_encoded'] = np.nan
for tr, val in kf.split(df):
    means = df.iloc[tr].groupby('city')['sales'].mean()
    df.loc[val, 'city_mean_encoded'] = df.iloc[val]['city'].map(means)
df['city_mean_encoded'].fillna(global_mean, inplace=True)


df['weekday_weekend_encoded'] = np.nan
for tr, val in kf.split(df):
    means = df.iloc[tr].groupby('weekday_weekend')['sales'].mean()
    df.loc[val, 'weekday_weekend_encoded'] = df.iloc[val]['city'].map(means)
df['weekday_weekend_encoded'].fillna(global_mean, inplace=True)

X_encoded = pd.concat([X_num, df[['product_mean_encoded', 'city_mean_encoded', "weekday_weekend_encoded"]]], axis=1)

C:\Users\Dalbir\AppData\Local\Temp\ipykernel_5304\3453947597.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['product_mean_encoded'].fillna(global_mean, inplace=True)
C:\Users\Dalbir\AppData\Local\Temp\ipykernel_5304\3453947597.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a 

## Correlation Based Feature Selection (Numeric)

In [112]:
corr = X_encoded.corrwith(y).sort_values(ascending=False)
print(corr[:15])

price_each              0.999202
product_mean_encoded    0.999202
SMA_3                   0.575449
SMA_5                   0.446030
CMA                     0.010487
hour                    0.001683
Lag_3                  -0.000265
day                    -0.000537
year                   -0.001006
Lag_5                  -0.001271
city_mean_encoded      -0.002571
week                   -0.003211
month                  -0.003454
quarter                -0.003483
quantity_ordered       -0.139564
dtype: float64


c:\Users\Dalbir\Downloads\Trends-Forecasting-Analytics-MLOps-Vertex-AI\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Dalbir\Downloads\Trends-Forecasting-Analytics-MLOps-Vertex-AI\venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


## F-regression (ANOVA F-test)

In [113]:
f_vals, p_vals = f_regression(X_encoded, y)
f_reg = pd.Series(f_vals, index=X_encoded.columns).sort_values(ascending=False)
print(f_reg.head(15))

price_each              1.162506e+08
product_mean_encoded    1.162411e+08
SMA_3                   9.192907e+04
SMA_5                   4.611464e+04
quantity_ordered        3.688607e+03
CMA                     2.042401e+01
quarter                 2.252005e+00
month                   2.214684e+00
week                    1.915081e+00
city_mean_encoded       1.227408e+00
hour                    5.260237e-01
Lag_5                   2.999662e-01
year                    1.877355e-01
day                     5.346414e-02
Lag_3                   1.308364e-02
dtype: float64


## Recursive Feature Elimination (model-driven)

In [114]:
model = LinearRegression()
rfe = RFE(model, n_features_to_select=30)
rfe.fit(X_encoded, y)
rfe_features = pd.Series(rfe.support_, index=X_encoded.columns)
selected_features = rfe_features[rfe_features == True].index.tolist()
print(selected_features)

['quantity_ordered', 'price_each', 'month', 'hour', 'year', 'week', 'day', 'quarter', 'SMA_3', 'SMA_5', 'Lag_3', 'Lag_5', 'CMA', 'product_mean_encoded', 'city_mean_encoded', 'weekday_weekend_encoded']


c:\Users\Dalbir\Downloads\Trends-Forecasting-Analytics-MLOps-Vertex-AI\venv\Lib\site-packages\sklearn\feature_selection\_rfe.py:300: UserWarning: Found n_features_to_select=30 > n_features=16. There will be no feature selection and all features will be kept.
  warnings.warn(


In [115]:
df.head()

,order_id,product,quantity_ordered,price_each,order_date,purchase_address,month,sales,city,hour,...,order_size,quarter,SMA_3,SMA_5,Lag_3,Lag_5,CMA,product_mean_encoded,city_mean_encoded,weekday_weekend_encoded
0,295665,Macbook Pro Laptop,1,1700.00,2019-12-30 00:01:00,"136 Church St, New York City, NY 10001",12,1700.00,New York City,0,...,single,4,0.000000,0.000,0.0,0.0,1700.000,1700.903534,187.536043,185.611936
1,295666,LG Washing Machine,1,600.00,2019-12-29 07:03:00,"562 2nd St, New York City, NY 10001",12,600.00,New York City,7,...,single,4,0.000000,0.000,0.0,0.0,1150.000,600.000000,188.305788,185.611936
2,295667,USB-C Charging Cable,1,11.95,2019-12-12 18:21:00,"277 Main St, New York City, NY 10001",12,11.95,New York City,18,...,single,4,770.650000,0.000,0.0,0.0,770.650,13.051909,188.305788,185.611936
3,295668,27in FHD Monitor,1,149.99,2019-12-22 15:13:00,"410 6th St, San Francisco, CA 94016",12,149.99,San Francisco,15,...,single,4,253.980000,0.000,1700.0,0.0,615.485,150.857998,184.813718,185.611936
4,295669,USB-C Charging Cable,1,11.95,2019-12-18 12:38:00,"43 Hill St, Atlanta, GA 30301",12,11.95,Atlanta,12,...,single,4,57.963333,494.778,600.0,0.0,494.778,13.088714,187.156659,185.611936


In [119]:
# List of selected features
selected_features = [
    'price_each',
    'product_mean_encoded',
    'city_mean_encoded',
    'SMA_3',
    'SMA_5',
    'quantity_ordered',
    'quarter',
    'month',
    'week',
    'year',
    'weekday_weekend_encoded'
]

# Keep only selected features + target
df_final = df[selected_features + ['sales']].copy()
output_path = r"C:/Users/Dalbir/Downloads/Trends-Forecasting-Analytics-MLOps-Vertex-AI/data/processed_file/sales_feature_selected.parquet"
# Save as Parquet
df_final.to_parquet(output_path, index=False)

print("File saved as 'data/sales_features_selected.parquet'")
print(df_final.head())
print(f"Shape after dropping unused columns: {df_final.shape}")


File saved as 'data/sales_features_selected.parquet'
   price_each  product_mean_encoded  city_mean_encoded       SMA_3    SMA_5  \
0     1700.00           1700.903534         187.536043    0.000000    0.000   
1      600.00            600.000000         188.305788    0.000000    0.000   
2       11.95             13.051909         188.305788  770.650000    0.000   
3      149.99            150.857998         184.813718  253.980000    0.000   
4       11.95             13.088714         187.156659   57.963333  494.778   

   quantity_ordered  quarter  month  week  year  weekday_weekend_encoded  \
0                 1        4     12     1  2020               185.611936   
1                 1        4     12    52  2019               185.611936   
2                 1        4     12    50  2019               185.611936   
3                 1        4     12    51  2019               185.611936   
4                 1        4     12    51  2019               185.611936   

     sales  
0 